In [ ]:
# NOTEBOOK NAME
# Histograms2D.ipynb
# NOTEBOOK NAME

# Opening Imports Section
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save 

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

In [ ]:
# REFLECTIVITY SENSITIVITY TO RANGE DIAGRAM
# this block plots histograms of reflectivity values at various ranges given a constant altitude
# this block works with GRIDDED radar data

# ADDED 2026-06-29 T 17:10K

# what does the radar data use for the "fill value"?
FillValue = -32

# CHOOSE ABSOLUTE OR RELATIVE frequencies to plot
AbsOrRel = 'Relative' # [must be exactly the strings: 'Absolute' or 'Relative']

# CHOOSE YOUR RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# CHOOSE YOUR ALTITUDE (must be a multiple of 500 m)
Altitude = 2500 # [m]

# set the spacing and range of ranges and reflectivities
DBZinterval = 1   # [dBZ]
RangeInterval = 10  # [km]

MaxRange = 150 * 1000 * (2)**(0.5) # [m] max range is 150 km in the diagonal (square root of 2)
MinRange = 0   # [m]

MaxDBZ =  60   # [dBZ]
MinDBZ = -10   # [dBZ]

NumRanges = int(np.ceil(MaxRange / (RangeInterval*1000) )) # the total number of range gates in consideration # convert from km to m
NumDBZs = int(np.ceil( (MaxDBZ-MinDBZ)/ (DBZinterval) )) # the total number of range gates in consideration # convert from km to m  

Ranges = np.arange(0,MaxRange*0.001,RangeInterval) # (X COORDINATE ON THE PLOT) create a list of all possible DBZ values from -39 to 100 in steps of 1 
DBZs = np.arange(MinDBZ, MaxDBZ, DBZinterval)  # (Y COORDINATE ON THE PLOT) create a list of all DBZ values to store data for

# name the radar site
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# CHOOSE YOUR DAY
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

# write out the data in one string
RadarFileDate  = YYYY + MM + DD 
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# create empty arrays for storing frequencies and relative of particular reflectivity occurences
RunningCountsGrid = np.full([NumRanges,NumDBZs], 0)
RunningNormCountsGrid = np.full([NumRanges,NumDBZs], 0)

# a running index of which time in the loop we are looking at
timei = 0
# LOOP OVER ALL SETS OF 5 MIN in the day
for houri in range(0,24):
    for mini in range(0,60,5):

        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 

        # save the first time used in the loop
        if (timei == 0):
            FirstFileTime = RadarFileTime
            FirstFileTimePrint = RadarFileTimePrint
        
        NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
        NetCDFstorageFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
        NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile 
                                
        try:
            xgrid = xr.open_dataset(NetCDFstoragePath)
            print('working on reading the file for ' + RadarFileTimePrint)
        except FileNotFoundError:
            print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
            continue

        # find the index in the file for the altitude you are considering
        AltI = np.where(xgrid.z == Altitude)[0][0]

        # calculate the distance from the radar and add it to the data frame as a new variable
        xgrid['distance'] = distance = np.sqrt(xgrid['x']**2 + xgrid['y']**2)
        xgrid['distance'].attrs = {'long_name': 'Horizontal distance from radar', 'units': 'm'}

        # loop through each range interval to calculate the DBZ histogram there
        for ri in range(0, NumRanges):
            
            # calculate the ranges of the annulus in consideration
            InnerRadius = ri * RangeInterval
            OuterRadius = (ri+1) * RangeInterval

            # mask the reflectivities to select only those in this particular range
            ConditionGridA = xgrid['distance'] > InnerRadius * 1000   # (y, x) boolean mask
            ConditionGridB = xgrid['distance'] < OuterRadius * 1000   # (y, x) boolean mask
            ConditionGrid = ConditionGridA * ConditionGridB    # combined boolean mask

            NumValidPoints = int(np.sum(ConditionGrid)) # number of grid points a valid DBZ value could be stored
            
            # only select the reflectivity for (the only time stored), the altitude you are looking at, and those masked ranges
            MaskedDBZs = xgrid['corrected_reflectivity'][0][AltI].where(ConditionGrid) 

            # take the DBZ array and unravel it and remove the NAN values
            MaskedDBZsFlat = np.array(MaskedDBZs).ravel()
            ValidDBZs = MaskedDBZsFlat[~np.isnan(MaskedDBZsFlat)]

            # retrieve counts/ relative counts of each reflectivity interval from the first output of np.histogram
            CountsGrid = np.histogram(ValidDBZs, bins = np.linspace(MinDBZ, MaxDBZ, NumDBZs + 1))[0]
            NormCountsGridHere = CountsGrid * (1/NumValidPoints) * 100

            # add to the running counts
            RunningNormCountsGrid[ri, :] = RunningNormCountsGrid[ri, :] + NormCountsGridHere
            RunningCountsGrid[ri, :] = RunningCountsGrid[ri, :] + CountsGrid
            
        timei = timei + 1

# retrieve the total number of times looped over
NumTimes = timei

# divide normalised counts grid by number of times to calculate mean occurances
PlottingNormCountsGrid = (RunningNormCountsGrid / NumTimes) * (1/DBZinterval)

# retrieve the last time used in the loop
LastFileTime = RadarFileTime
LastFileTimePrint = RadarFileTimePrint

# # normalise frequencies to per unit DBZ per km of height
PlottingCountsGrid = RunningCountsGrid * (1/DBZinterval) * (1/RangeInterval) * (1/NumTimes)

# MAKE THE PLOT
fig, ax = plt.subplots(figsize=(8,6))

if (AbsOrRel == 'Absolute'):
    MaxFreq = 30
    ColorBarLabel = 'Counts [per DBZ per km per 5 min]'
    PlotArray = PlottingCountsGrid
elif (AbsOrRel == 'Relative'):
    MaxFreq = 10
    ColorBarLabel = 'Relative Counts [% Values at each Range per dBZ]'
    PlotArray = PlottingNormCountsGrid

CFAD1 = pcolormeshC(DBZs, Ranges, PlotArray, ax=ax, cmap='nipy_spectral', vmin=0, vmax=MaxFreq)
plt.colorbar(CFAD1, ax=ax, label = ColorBarLabel)

plt.xlim([0, MaxDBZ-(DBZinterval*0.5)]) # cut out the last half of the last DBZ column to crop to where the values are plotted
plt.ylim([0, (MaxRange*0.001)])

plt.grid()

plt.title(AbsOrRel + ' Corrected Reflectivity Values by Range\nAt ' + str(Altitude*0.001) + ' km Altitude for ' + \
          RadarSiteName + ' Radar\n' +  'Between ' + \
          FirstFileTimePrint + ' and ' + LastFileTimePrint + ' UTC on ' + RadarFileDatePrint)

ax.set_xlabel('Reflectivity [dBZ]')
ax.set_ylabel('Range [km]')

PlotVar  = 'corrected_reflectivity'

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/Sensitivity/' + RadarIDno + '/' + RadarFileDate + '/'
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + FirstFileTime + 'to' + LastFileTime + \
'_' + AbsOrRel + '_' + PlotVar + '_Sensitivity.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('Creating Folder: ' + SaveFolder)
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
plt.close()

In [ ]:
# Z by RANGE "FAKE CFAD"
# this block plots the mean reflectivies by range and altitude for gridded data
# I think right now it is set up for just a 5 minute periodL

# ADDED 2026-06-10T18:56UTC+10:00

# what does the radar data use for the "fill value"?
FillValue = -32

RangeInterval = 1 # [km] increments of range in the plot

# CHOOSE YOUR RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# name the radar site
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# CHOOSE YOUR DAY
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

# CHOOSE YOUR HEIGHTS OF CONSIDERATION
MinHeight = float(np.min(xgrid.z)*0.001) # [km] minimum height in the plot
MaxHeight = float(np.max(xgrid.z)*0.001) # [km] maximum height in the plot

# LOOP OVER ALL SETS OF 5 MIN in the day
for houri in range(20,21):
    for mini in range(0,5,5):

        # add leading zeros for strings
        YYYY = str(RadarYear).zfill(4)
        MM = str(RadarMonth).zfill(2)
        DD = str(RadarDay).zfill(2)
        
        # write out the data in one string
        RadarFileDate  = YYYY + MM + DD 
        
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
        
        NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
        NetCDFstorageFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
        NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile 
                                
        try:
            xgrid = xr.open_dataset(NetCDFstoragePath)
            print('working on reading the file for ' + RadarFileTimePrint)
        except FileNotFoundError:
            print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
            continue

        # calculate the distance from the radar and add it to the data frame as a new variable for data selection purposes
        xgrid['distance'] = distance = np.sqrt(xgrid['x']**2 + xgrid['y']**2)
        xgrid['distance'].attrs = {'long_name': 'Horizontal distance from radar', 'units': 'm'}
        
        ConsideredHeights = xgrid.z[ np.where( (xgrid.z<=MaxHeight*1000) & (xgrid.z>=MinHeight*1000) ) ]
        
        NumHeights = np.size(ConsideredHeights) # the total number of altitudes in consideration
        
        # # find a floor and ceiling to the values of DBZ
        # LoEndDBZ = MinDBZ #int(np.floor(np.nanmin(flatZs)))
        # HiEndDBZ = MaxDBZ #int(np.ceil(np.nanmax(flatZs)))

        MaxRange = np.array(np.max(xgrid.distance))
        MinRange = 0
        
        NumRanges = int(np.ceil(MaxRange / (RangeInterval*1000) )) # the total number of range gates in consideration # convert from km to m
        
        # create an empty array for storing frequencies and relative frequencies
        NormCountsGrid = np.full([NumRanges,NumHeights], np.nan)

        
        # loop through each range range and height
        for ri in range(0, NumRanges):
            # print('working on range ' + str(ri))
            # calculate the ranges of the annulus in consideration
            InnerRadius = ri * RangeInterval
            OuterRadius = (ri+1) * RangeInterval

            # mask the reflectivities 
            ConditionGridA = xgrid['distance'] > InnerRadius * 1000   # (y, x) boolean mask
            ConditionGridB = xgrid['distance'] < OuterRadius * 1000   # (y, x) boolean mask
            ConditionGrid = ConditionGridA * ConditionGridB    # combined boolean mask
            masked_reflectivity = xgrid['corrected_reflectivity'].where(ConditionGrid)  # MASK THE Z VALUES
            
            for zi in range(0, NumHeights):
                NormCountsGrid[ri,zi] = np.nanmean(masked_reflectivity[0][zi])
                
        
        # intervals of DBZ bins on the plot
        DBZspacing = 1 # [dBZ]
        
        Ranges = np.arange(0,MaxRange*0.001,RangeInterval) # (X COORDINATE ON THE PLOT) create a list of all possible DBZ values from -39 to 100 in steps of 1 
                                     # convert from m to km
        Heights = np.array(ConsideredHeights) * 0.001  # (Y COORDINATE ON THE PLOT) [converted to km] create a list of all of the altitudes where data are stored
        
        # intervals of altitudes on the plot
        HeightSpacing = 0.5 # [km] 
        
        UprightFrequencies = np.transpose(NormCountsGrid) # (VALUES ON THE PLOT) transpose the stored relative frequencies of the reflectivities
        
        # # normalise frequencies to per unit DBZ per km of height
        # NormUprightFrequencies = UprightFrequencies * (1/DBZspacing) * (1/HeightSpacing)  
        
        # PLOT A CFAD! (sort of)
        fig, ax = plt.subplots(figsize=(8,6))
        
        CFAD1 = pcolormeshC(Ranges , Heights, UprightFrequencies, ax=ax, cmap='nipy_spectral', vmin=0, vmax=40)
        plt.colorbar(CFAD1, ax=ax, label = 'Mean dBZ')
        
        plt.xlim([0, (MaxRange*0.001) + 10])
        plt.ylim([0, MaxHeight+1])

        plt.grid()
        
        plt.title('Mean Corrected Reflectivity Value by Altitude and Range\n for ' + RadarSiteName + ' Radar on ' + \
                  RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
                  str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
        
        ax.set_xlabel('Range [km]')
        ax.set_ylabel('Altitude [km]')
        
        PlotVar  = 'corrected_reflectivity'
        
        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/MeanZ/' + RadarIDno + '/' + RadarFileDate + '/'
        SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_MeanZ.png'
        
        SavePath = SaveFolder + SaveFile
        
        if not Path(SaveFolder).exists():
            print('Creating Folder: ' + SaveFolder)
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
        # plt.close()